# W6D4 — Efficient LLM Adaptation: LoRA Fine-Tune + 4-bit Quantization — Guided

**Week 6 · Day 4 · LLMs** · Lab

> **The big idea.** You almost never fine-tune every weight of an LLM. Instead
> you **freeze the base model** and train a tiny set of extra matrices — a
> **LoRA adapter** — that steer the frozen weights toward your task. Then you
> **quantize** the frozen weights to 4-bit so inference costs a quarter of the
> memory, with almost no quality loss. The two tricks together turn
> "unaffordable" into "runs on Colab free tier".

In the first three days you probed, retrieved, and used LLMs. Today you
**change** one — teach a 500M-parameter base model to answer in the voice of
the Olo Academy handbook, then slim it down to 4-bit for cheap deployment. By
the end you'll have numbers: trainable parameters (~0.2%), training time
(~2-3 min on T4), memory before/after quantization, and tokens/second.

<div dir="rtl" align="right">

> **الفكرة الكبرى.** نادرًا ما نُدرّب كل أوزان نموذج لغويّ. بل **نُجمِّد النموذج
> الأساس** وندرّب مجموعة صغيرة من المصفوفات الإضافيّة — **محوِّل LoRA** — تُوجِّه
> الأوزان المُجمَّدة نحو مهمّتك. ثم **نُكمّم** الأوزان المُجمَّدة إلى ٤ بت ليكلّف
> الاستدلال رُبع الذاكرة بلا خسارة تُذكَر. الحيلتان معًا تحوّلان "غير مُستطاع"
> إلى "يعمل على طبقة Colab المجانيّة".

في الأيّام الثلاثة الماضية فحصت واسترجعت واستخدمت نماذج. اليوم **تُغيّر** نموذجًا
— تعلّم نموذج ٥٠٠M معامل أن يُجيب بصوت دليل أكاديمية أولو، ثم تُنحّفه إلى ٤ بت.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

Concretely, you can:

- Format a tiny instruction dataset into the exact token stream the model
  expects for **supervised fine-tuning (SFT)**.
- Configure a **LoRA adapter** on the right modules (q/k/v projections), count
  the trainable parameters, and show they are ~0.1–0.5% of the base model.
- Fine-tune the adapter for a couple of epochs and compare its answers to the
  baseline, side by side, on the same questions.
- Load the fine-tuned model with **4-bit quantization** (QLoRA) and measure the
  **memory drop** (FP32 → 4-bit is roughly 8x).
- Describe the trade-off between fine-tune accuracy, inference cost, and
  memory footprint for a small production model.

<div dir="rtl" align="right">

## الأهداف التعلّمية

- أن تُنسِّق مجموعة تعليميّة صغيرة إلى تدفّق الرموز الذي يتوقّعه النموذج للـ**ضبط
  الدقيق المُشرَف (SFT)**.
- أن تُهيّئ **محوِّل LoRA** على الوحدات الصحيحة، تعدّ المعاملات القابلة للتدريب،
  وتُثبت أنّها نحو ٠٫١–٠٫٥٪ من النموذج الأساس.
- أن تدرّب المحوِّل لحقبتين، وتقارن إجاباته مع الأساس على نفس الأسئلة.
- أن تحمّل النموذج بـ**تكميم ٤ بت** (QLoRA) وتقيس انخفاض الذاكرة.
- أن تصف المقايضة بين الدقّة بعد الضبط، وتكلفة الاستدلال، وبصمة الذاكرة.

</div>


## About the model and data

**Base model:** `Qwen/Qwen2.5-0.5B` — 494M parameters, open base model (**not**
instruction-tuned), bilingual EN/AR. Small enough that a LoRA fine-tune
finishes in minutes on a free Colab T4. We use the base, not the Instruct
variant, so the fine-tuning effect is dramatic and visible: the base continues
text generically; the LoRA-tuned version answers your questions in the exact
voice of your training data.

**Adapter config:** LoRA on `q_proj` and `v_proj` of every attention layer,
rank `r=8`, `alpha=16`, `dropout=0.05`. That is ~1M trainable parameters —
**roughly 0.2% of the base model**. The full adapter is a few MB on disk.

**Quantization:** 4-bit NF4 via `bitsandbytes`, with a second quantization
pass on the quantization constants (`use_double_quant=True`).

**Data:** **10 Olo-Academy FAQ pairs** — the same handbook from D2 reshaped
into `Question → Answer` form. Hand-written, inline in the notebook, no
download. Tiny on purpose: today's point is that LoRA is *so* parameter-
efficient that a handful of examples teaches a 500M-param model a specific
behaviour. In production you'd scale this to hundreds or thousands of pairs
(SFT), with the same code.

**GPU required.** `bitsandbytes` needs CUDA. The whole lab runs on **Colab T4
free tier** in under 10 minutes end-to-end.

<div dir="rtl" align="right">

## عن النموذج والبيانات

**النموذج الأساس:** `Qwen/Qwen2.5-0.5B` — ٤٩٤M معامل، أساس مفتوح (**غير**
مُدرَّب على التعليمات)، ثنائيّ اللغة. صغير بما يكفي ليُنهي ضبط LoRA في دقائق على
Colab T4 مجاني. استخدمنا الأساس لا Instruct ليظهر أثر الضبط بوضوح.

**إعداد المحوِّل:** LoRA على `q_proj` و`v_proj` في كلّ طبقة انتباه، رتبة `r=8`،
`alpha=16`، `dropout=0.05`. يعادل ~١M معامل قابل للتدريب — **نحو ٠٫٢٪ من
الأساس**. المحوّل الكامل بضع ميجابايتات.

**التكميم:** ٤ بت NF4 عبر `bitsandbytes` مع تكميم ثانٍ على ثوابت التكميم.

**البيانات:** **١٠ أزواج أسئلة وأجوبة** عن أكاديمية أولو — نفس دليل D2
معاد تشكيله كـ`Question → Answer`. مكتوبة يدويًّا، داخل الخلايا، بلا تنزيل.
صغيرة عمدًا: درس اليوم أن LoRA كفؤ *جدًّا* بالمعاملات، فحفنة أمثلة تُعلِّم
نموذج ٥٠٠M معامل سلوكًا محدّدًا. في الإنتاج تُوسِّعها لمئات أو آلاف من الأزواج.

**GPU مطلوب.** يعمل الدرس كاملًا على Colab T4 المجاني في أقل من ١٠ دقائق.

</div>


## Setup

<div dir="rtl" align="right">

## الإعداد

</div>


In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformers>=4.40", "accelerate",
                           "peft>=0.11", "bitsandbytes>=0.43", "datasets"])

import torch, time, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel

assert torch.cuda.is_available(), "This lab needs a GPU. Switch Colab runtime to T4."
DEVICE = "cuda"
SEED = 42; torch.manual_seed(SEED)
print(f"torch={torch.__version__}  gpu={torch.cuda.get_device_name()}")
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
# === Load the base model (float16 on GPU) ==========================================
BASE_NAME = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(BASE_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_NAME, torch_dtype=torch.float16, device_map="cuda",
)
base_model.eval()
total_params = sum(p.numel() for p in base_model.parameters()) / 1e6
print(f"Loaded {BASE_NAME}: {total_params:.0f}M params in fp16")

## Section 1 — Warm-up: baseline behaviour  *(~10 min)*

Nothing to implement. Watch the base (not instruction-tuned, not fine-tuned)
model answer three Olo-Academy questions. It will **not** know specific facts
(cohort size, instructor name, etc.) — it will complete the prompt in a
general way. Save these three outputs; you will compare them to the LoRA'd
model's answers in Section 2.

<div dir="rtl" align="right">

## القسم ١ — إحماء: سلوك الأساس  *(≈١٠ دقائق)*

لا شيء تنفّذه. راقب النموذج الأساس (غير مُدرَّب على التعليمات، وغير مُضبوط) وهو
يُجيب على ثلاثة أسئلة. **لن** يعرف الحقائق الخاصّة. احفظ المخرجات لمقارنتها بعد
LoRA.

</div>


In [ ]:
# === Helper: one-shot answer from any causal LM ====================================
@torch.no_grad()
def answer(model, prompt, max_new_tokens=60):
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip()

QUESTIONS = [
    "Question: How many weeks does the AIEP program last?\nAnswer:",
    "Question: Who is the lead instructor of the AIEP program?\nAnswer:",
    "Question: What is required to receive the AIEP certificate?\nAnswer:",
]

baseline_answers = []
for q in QUESTIONS:
    a = answer(base_model, q)
    baseline_answers.append(a)
    print(q)
    print(" ", a, "\n")

## Section 2 — Fine-tune with LoRA  *(~55 min)*

Five tasks:

1. **Format the dataset** — turn 10 Olo-Academy FAQ pairs into the exact
   `Question: ... Answer: ...` strings the model will learn to continue.
2. **Configure LoRA** — pick which modules to adapt and the rank, print the
   trainable-parameter count.
3. **Train** — one `Trainer.train()` call, 3 epochs, ~2 minutes on T4.
4. **Compare** — ask the same three Section-1 questions to the LoRA'd model
   and read the difference side-by-side.
5. **Count parameters** — print trainable vs base ratios to see how little
   you actually changed.

<div dir="rtl" align="right">

## القسم ٢ — ضبط بـLoRA  *(≈٥٥ دقيقة)*

خمس مهام: نسّق البيانات، هيّئ LoRA، درّب، قارن قبل/بعد، احسب نسبة المعاملات.

</div>


### Task 1 — Format the tiny Olo-Academy FAQ dataset

Below is a Python list of 10 Q&A pairs. Build a Hugging Face `Dataset` of
single-string examples in the exact format
`"Question: {q}\nAnswer: {a}" + eos_token`
so the model learns to continue a question into an answer. Tokenize it with
truncation + padding, and set `labels = input_ids` so the Trainer computes
the standard causal-LM loss.

<div dir="rtl" align="right">

### المهمّة ١ — نسّق مجموعة الأسئلة والأجوبة

نسّق قائمة ١٠ أزواج Q/A إلى سلاسل بتنسيق واحد، وضمّنها في `Dataset`. اضبط
`labels = input_ids` ليحسب Trainer خسارة causal-LM القياسيّة.

</div>


In [ ]:
# === The tiny FAQ dataset (keep-all) =============================================
from datasets import Dataset

FAQ = [
    ("How many weeks does the AIEP program last?",
     "The AIEP program lasts 8 weeks."),
    ("Who is the lead instructor of the AIEP program?",
     "The lead instructor of the AIEP program is Ghaida Alshehri."),
    ("What is required to receive the AIEP certificate?",
     "To receive the AIEP certificate you must submit all 5 weekly projects and the capstone with a passing grade."),
    ("What is the maximum cohort size for October 2026?",
     "The October 2026 cohort is capped at 40 students."),
    ("Where does the AIEP program take place?",
     "The AIEP program takes place at Olo Academy in Riyadh, Saudi Arabia."),
    ("When does the AIEP capstone presentation happen?",
     "The AIEP capstone presentation happens on the final Thursday of week 8."),
    ("How many hours of self-study per week are expected?",
     "Students are expected to complete 20 hours of self-study per week."),
    ("What is the main peer-support channel for AIEP?",
     "The main peer-support channel for AIEP is the Olo Academy Discord server."),
    ("What are the office hours of the lead AIEP instructor?",
     "The lead instructor holds office hours on Tuesday and Thursday from 2 PM to 4 PM Riyadh time."),
    ("What is the attendance policy of AIEP?",
     "Attendance below 70% of live sessions results in automatic removal from the program."),
]
print(f"Loaded {len(FAQ)} FAQ pairs.")

In [ ]:

def format_example(q, a):
    return f"Question: {q}\nAnswer: {a}" + tokenizer.eos_token

# TODO: Turn FAQ into a tokenized Dataset with input_ids, attention_mask, labels.
# مهمة: حوّل FAQ إلى Dataset بـ input_ids و attention_mask و labels.

print(train_ds)
print("\nFirst tokenised example:")
print(f"  input_ids length: {len(train_ds[0]['input_ids'])}")
print(f"  first 15 tokens: {train_ds[0]['input_ids'][:15]}")

### Task 2 — Configure LoRA and wrap the base model

Build a `LoraConfig` targeting `q_proj` and `v_proj` with `r=8`, `alpha=16`,
`dropout=0.05`, and `task_type="CAUSAL_LM"`. Wrap the base model with
`get_peft_model`. Then call `.print_trainable_parameters()` to see the
trainable count and the ratio.

<div dir="rtl" align="right">

### المهمّة ٢ — هيّئ LoRA ولفّ النموذج

ابنِ `LoraConfig` يستهدف `q_proj` و`v_proj` بـ`r=8`، `alpha=16`،
`dropout=0.05`، و`task_type="CAUSAL_LM"`. لفّ النموذج بـ`get_peft_model`.
اطبع عدد المعاملات القابلة للتدريب.

</div>


In [ ]:

# TODO: Build LoraConfig, wrap base_model with get_peft_model, print trainable params.
# مهمة: ابنِ LoraConfig، لفّ base_model بـget_peft_model، اطبع عدد المعاملات.

### Task 3 — Train the adapter

Configure the `Trainer` with a small learning rate (`2e-4`), 3 epochs,
batch size 2, no evaluation (we don't have a held-out set). Call `train()`.
Expect ~2 minutes on T4 and a loss that drops from ~3 to below 0.5.

<div dir="rtl" align="right">

### المهمّة ٣ — درّب المحوِّل

اضبط `Trainer` بـلر `2e-4`، ٣ حقبات، حجم دفعة ٢، بلا تقييم. نادِ `train()`.

</div>


In [ ]:

# TODO: Train the LoRA adapter for 3 epochs with Trainer.
# مهمة: درّب المحوّل ٣ حقبات بـTrainer.

### Task 4 — Compare baseline vs LoRA on the same questions

Ask the LoRA'd model (now `peft_model`) the same three questions you saved in
Section 1 and print the before/after side by side. Expect the LoRA'd answers
to be short, specific, and match the handbook.

<div dir="rtl" align="right">

### المهمّة ٤ — قارن قبل/بعد

اسأل peft_model نفس الأسئلة الثلاثة واطبع قبل/بعد. توقّع إجابات محدّدة ومطابقة
للدليل.

</div>


In [ ]:

# TODO: Compare baseline vs LoRA answers on the same three questions.
# مهمة: قارن إجابات الأساس وLoRA على نفس الأسئلة الثلاثة.

### Task 5 — Count parameters: base vs trainable vs LoRA adapter

Print three numbers:
- Total parameters of the base model.
- Trainable parameters (just the LoRA matrices).
- Ratio of trainable / total — expect **~0.2%** or less.

The point: you got a task-specific model by training ~1M parameters instead
of 494M, and the full adapter would fit in a few MB.

<div dir="rtl" align="right">

### المهمّة ٥ — عدّ المعاملات

اطبع ثلاثة أرقام: الكلّي، القابل للتدريب، والنسبة. المحوّل الكامل ~١M معامل
بدلًا من ٤٩٤M.

</div>


In [ ]:

# TODO: Print total, trainable, and trainable/total ratio for peft_model.
# مهمة: اطبع الكلّي والقابل للتدريب والنسبة لـpeft_model.

### Takeaway

You just taught a 494M-param model to answer in the voice of a specific
corpus by training **less than 1M parameters** on **10 examples**. The LoRA
adapter is a few MB on disk, swappable at serve time, and composable (you
could load several and switch between them for different tenants).

This is why every modern LLM product you use is secretly **a base model +
many LoRA adapters** — one for each customer, one for each language, one for
each task — not a fleet of fully-tuned giants.

<div dir="rtl" align="right">

### الخلاصة

درّبت نموذج ٤٩٤M معامل ليُجيب بصوت مجموعة محدّدة بتدريب أقلّ من ١M معامل على
١٠ أمثلة فقط. محوّل LoRA بضع ميجابايتات، قابل للتبديل وقت الخدمة، وقابل
للتركيب. ولهذا كل منتج LLM حديث هو سرًّا **نموذج أساس + محوّلات LoRA متعدّدة**.

</div>


## Section 3 — Make it cheap: 4-bit quantization  *(~25 min)*

Fine-tuning changed what the model *says*. Quantization changes what the
model *costs*. Two tasks + write-up:

1. **Load the base model in 4-bit** (NF4 via `bitsandbytes`) and measure the
   memory drop vs the fp16 version you have been using.
2. **Compare generation latency** fp16 vs 4-bit on the same prompt. On T4 the
   4-bit version is often similar or faster for small models — but the real
   win is the memory, which is what lets 7B/13B models fit at all.

<div dir="rtl" align="right">

## القسم ٣ — جعله رخيصًا: تكميم ٤ بت  *(≈٢٥ دقيقة)*

الضبط غيّر ما *يقوله* النموذج. التكميم يُغيّر ما *يُكلّفه*. مهمّتان + كتابة.

</div>


In [ ]:
# === Record FP16 memory footprint (keep-all) =======================================
torch.cuda.empty_cache(); gc.collect()
mem_fp16 = base_model.get_memory_footprint() / 1e6
print(f"Base model (fp16) memory footprint: {mem_fp16:.1f} MB")

### Task 1 — Reload the base model in 4-bit and measure memory

Build a `BitsAndBytesConfig` with `load_in_4bit=True`, `bnb_4bit_quant_type="nf4"`,
`bnb_4bit_compute_dtype=torch.float16`, `bnb_4bit_use_double_quant=True`.
Reload the model with that config and print its memory footprint.

<div dir="rtl" align="right">

### المهمّة ١ — أعِد تحميل الأساس بـ٤ بت وقِس الذاكرة

ابنِ `BitsAndBytesConfig` بالإعدادات، أعِد التحميل، اطبع بصمة الذاكرة.

</div>


In [ ]:

# TODO: Build the 4-bit config, reload the model, print the memory drop vs FP16.
# مهمة: ابنِ إعداد ٤ بت، أعِد التحميل، اطبع فرق الذاكرة.

### Task 2 — Compare generation latency: FP16 vs 4-bit

For a fixed prompt and the same number of generated tokens, measure
wall-clock time for both models. Warm up with one generation each (first call
compiles kernels), then time 3 generations and average.

<div dir="rtl" align="right">

### المهمّة ٢ — قارن زمن التوليد: FP16 مقابل ٤ بت

لمُحثّ ثابت وعدد رموز ثابت، قس الزمن للنموذجين. سخّن بتوليد واحد لكلٍّ، ثم
قس ٣ توليدات ومتوسّط.

</div>


In [ ]:

PROMPT = "Question: What is the capital of Saudi Arabia?\nAnswer:"
N = 3

# TODO: Time FP16 vs 4-bit over 3 generations each, after one warmup each.
# مهمة: قس FP16 مقابل ٤-بت على ٣ توليدات بعد تسخين واحد لكلٍّ.

### Write-up

In **one English sentence**, write down when 4-bit quantization is a slam-dunk
win and when it is not worth the small accuracy loss. (Hint: think about model
size, VRAM budget, and whether you need the last 1-2% of accuracy.)

> *Example answer: "4-bit quantization is a slam-dunk whenever VRAM is the
> bottleneck — a 7B model in 4-bit fits where a 7B in fp16 does not — but it
> is not worth the slight quality drop for already-small models that fit
> comfortably in fp16 and are latency-bound rather than memory-bound."*

<div dir="rtl" align="right">

### كتابة

في جملة إنجليزيّة واحدة، متى يكون تكميم ٤-بت ربحًا مؤكّدًا ومتى لا يستحقّ؟

</div>


## Wrap-up

Four things you can now do:

1. **Fine-tune any open LLM with LoRA** on your own small dataset in under 5
   minutes on free hardware.
2. **Count** trainable vs base parameters and report the ratio (expect ~0.2%
   with r=8 on q/v projections).
3. **Quantize** a model to 4-bit with NF4 and measure the memory + latency
   trade-off end-to-end.
4. **Decide** between fine-tuning, RAG, and prompting for a new task,
   knowing roughly what each costs.

This closes the LLM arc of Week 6: you can **probe** (D1), **ground** (D2),
**act with** (D3), and **cheaply adapt** (D4) any open LLM. **D5** is either
a Kaggle-style challenge that ties them together or an intro to Vision-Language
Models — decided based on class pacing.

<div dir="rtl" align="right">

## الخلاصة

أربعة أمور تستطيع الآن: ضبط دقيق بـLoRA على بياناتك في دقائق، عدّ المعاملات
القابلة للتدريب، التكميم إلى ٤ بت، واختيار بين الضبط وRAG والمُحثّ. يختم هذا
قوس LLM: تفحص (D1)، ترسّخ (D2)، تتصرّف (D3)، وتُكيّف برخص (D4).

</div>
